Nama : Yesha Faradina

NIM : 240401010072

Kelas : IF403

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk  = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
           'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

transaksi = []
for _ in range (50):
    n_item  =  np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
      transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [2]:
from mlxtend.preprocessing import TransactionEncoder

te      = TransactionEncoder()
te_ary  = te.fit(transaksi).transform(transaksi)
df      = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [11]:
import warnings
warnings.filterwarnings("ignore")

from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')
freq_items  = apriori(df, min_support=0.1, use_colnames=True)
freq_items  = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))


min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


In [8]:
import warnings
warnings.filterwarnings("ignore")

from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence',
                          min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))


         antecedents consequents  support  confidence      lift
9        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
15  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
12      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
13     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
8      (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
11     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
14   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


Aturan terkuat berdasarkan lift tertinggi adalah Teh & Keju → Telur (2,38). Dari perspektif bisnis, aturan seperti Roti → Selai lebih intuitif untuk rekomendasi produk, penjualan silang, dan tata letak toko, karena mencerminkan produk yang umumnya dibeli pelanggan secara bersamaan.

In [9]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy', 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix  = cosine_similarity(fitur)
def rekomendasi_serupa(nama_produk, top_n=3):
  idx   = katalog.index[katalog['produk'] == nama_produk][0]
  skor  = list(enumerate(sim_matrix[idx]))
  skor = sorted(skor, key=lambda x: x[1], reverse=True)
  skor = [s for s in skor if s[0] != idx][:top_n]
  return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [10]:
import warnings
warnings.filterwarnings("ignore")

produk_target = 'Roti'

rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules: ')
print(rules_terkait[['consequents', 'lift']].head())

print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules: 
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


1. Apakah kedua pendekatan tersebut konsisten?

Ya, keduanya konsisten. Baik Association Rules maupun Content-Based Filtering merekomendasikan Selai bagi pelanggan yang tertarik pada Roti. Hal ini menunjukkan bahwa kedua metode mengidentifikasi selai sebagai rekomendasi yang sangat relevan. Selain itu, pendekatan berbasis konten menyarankan Serealdan Susu karena memiliki karakteristik produk yang serupa, meskipun produk-produk tersebut tidak sering dibeli bersamaan dengan roti.

2. Kapan sebaiknya setiap pendekatan digunakan?

Association Rules: Paling tepat digunakan ketika tersedia data transaksi yang memadai dan tujuannya adalah merekomendasikan produk yang sering dibeli bersamaan.
Content-Based Filtering: Paling tepat digunakan ketika tersedia fitur atau atribut produk, khususnya untuk merekomendasikan produk serupa atau saat riwayat transaksi terbatas.


3. Kapan sebaiknya keduanya digabungkan (Hybrid)?

Pendekatan hybrid umumnya merupakan pilihan terbaik karena menggabungkan kelebihan dari kedua metode tersebut. Association Rules menangkap perilaku pembelian pelanggan yang sebenarnya, sementara Content-Based Filtering merekomendasikan produk serupa, sehingga menghasilkan rekomendasi yang lebih akurat, beragam, dan dapat diandalkan.

